# Titanic Survival Prediction | Machine Learning Project

## Objective

Predict whether a passenger survived the Titanic disaster using machine learning.

This project demonstrates an end-to-end ML workflow including:
- Data exploration
- Data cleaning
- Feature engineering
- Model training
- Model evaluation
- Kaggle submission

## Tools Used
- Python
- Pandas
- Scikit-learn
- Matplotlib

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load Dataset

We load the Titanic dataset using Pandas.

In [2]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

# Exploratory Data Analysis (EDA)

We explore the dataset to understand:
- Data structure
- Missing values
- Data types
- Basic statistics

In [3]:
print(train.head())
print(train.info())
print(train.isnull().sum())
print(train.duplicated().sum())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

# Data Cleaning

We handle missing values and prepare the dataset for machine learning.

In [4]:
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(test['Age'].median())

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])

test['Fare'] = test['Fare'].fillna(test['Fare'].median())

train.drop(columns=['Cabin'], inplace=True)
test.drop(columns=['Cabin'], inplace=True, errors='ignore')

# Categorical Encoding

We convert categorical variables into numerical format so machine learning models can process them.

In [5]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)
test = pd.get_dummies(test, columns=['Embarked'], drop_first=True)

# Feature Engineering

We create new features to improve model performance:
- FamilySize
- IsAlone
- Title

In [6]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

In [7]:
train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']

for df in [train, test]:
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    df['Title'] = df['Title'].replace(['Mlle','Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

le = LabelEncoder()
le.fit(pd.concat([train['Title'], test['Title']]))

train['Title'] = le.transform(train['Title'])
test['Title'] = le.transform(test['Title'])

# Feature Selection

We select the most relevant features for training the model.

In [8]:
features = [
    'Pclass','Sex','Age','Fare',
    'Embarked_Q','Embarked_S',
    'FamilySize','IsAlone','Title'
]

X = train[features]
y = train['Survived']

# Train-Test Split

We split the data into training and validation sets to evaluate performance.

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Logistic Regression Model

We train a simple baseline model for comparison.

In [10]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_val)
log_acc = accuracy_score(y_val, log_pred)

print("Logistic Regression Accuracy:", log_acc)

Logistic Regression Accuracy: 0.8100558659217877


# Random Forest Parameters

We test different Random Forest settings to find a good balance between performance and overfitting.

We tune two important parameters:

- n_estimators (number of trees)
- max_depth (tree depth)

Higher values usually improve performance but increase overfitting risk.

In [11]:
best_score = 0
best_params = {}

for n in [100, 200, 300]:
    for d in [5, 7, 9, 12]:
        
        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=d,
            random_state=42,
            class_weight='balanced'
        )
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        
        acc = accuracy_score(y_val, preds)
        
        print(f"n_estimators={n}, max_depth={d}, accuracy={acc:.4f}")
        
        if acc > best_score:
            best_score = acc
            best_params = {'n_estimators': n, 'max_depth': d}

print("\nBest Parameters:", best_params)
print("Best Accuracy:", best_score)

n_estimators=100, max_depth=5, accuracy=0.8268
n_estimators=100, max_depth=7, accuracy=0.8268
n_estimators=100, max_depth=9, accuracy=0.8547
n_estimators=100, max_depth=12, accuracy=0.8436
n_estimators=200, max_depth=5, accuracy=0.8156
n_estimators=200, max_depth=7, accuracy=0.8380
n_estimators=200, max_depth=9, accuracy=0.8547
n_estimators=200, max_depth=12, accuracy=0.8492
n_estimators=300, max_depth=5, accuracy=0.8268
n_estimators=300, max_depth=7, accuracy=0.8380
n_estimators=300, max_depth=9, accuracy=0.8603
n_estimators=300, max_depth=12, accuracy=0.8380

Best Parameters: {'n_estimators': 300, 'max_depth': 9}
Best Accuracy: 0.8603351955307262


# Random Forest Model

We train a Random Forest model to capture non-linear relationships.

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)
rf_acc = accuracy_score(y_val, rf_pred)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.8603351955307262


# Model Selection

We compare the performance of Logistic Regression and Random Forest using validation accuracy.

The model with the higher accuracy is selected as the final model for prediction.

This ensures that our final predictions are based on the best-performing model on unseen data.

In [13]:
# Store accuracies in a simple comparison table
model_results = {
    "Logistic Regression": log_acc,
    "Random Forest": rf_acc
}

for model_name, acc in model_results.items():
    print(f"{model_name}: {acc:.4f}")

# Select best model
if rf_acc > log_acc:
    best_model = rf_model
    best_model_name = "Random Forest"
else:
    best_model = log_model
    best_model_name = "Logistic Regression"

print("\nSelected Model:", best_model_name)

Logistic Regression: 0.8101
Random Forest: 0.8603

Selected Model: Random Forest


# Final Model

The selected model is now used for predictions on the test dataset.

In [14]:
# Store accuracies in a simple comparison table
model_results = {
    "Logistic Regression": log_acc,
    "Random Forest": rf_acc
}

for model_name, acc in model_results.items():
    print(f"{model_name}: {acc:.4f}")

# Select best model
if rf_acc > log_acc:
    best_model = rf_model
    best_model_name = "Random Forest"
else:
    best_model = log_model
    best_model_name = "Logistic Regression"

print("\nSelected Model:", best_model_name)

Logistic Regression: 0.8101
Random Forest: 0.8603

Selected Model: Random Forest


# Predict on Validation Data

We use the trained model to make predictions on the validation set.

This helps us evaluate how well the model performs on unseen data before generating final predictions.

In [15]:
y_pred = rf_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_val, y_pred))

Accuracy: 0.8603351955307262

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.90      0.88       105
           1       0.85      0.81      0.83        74

    accuracy                           0.86       179
   macro avg       0.86      0.85      0.86       179
weighted avg       0.86      0.86      0.86       179


Confusion Matrix:

[[94 11]
 [14 60]]


# Predict on the Real Test Set (Kaggle Submission)

We now use the trained model to predict survival on the test dataset.

The output is formatted according to Kaggle requirements and saved as a submission file.

In [16]:
# Select same features used in training
X_test = test[features]

# Predict survival
test['Survived'] = rf_model.predict(X_test)

# Create submission file
submission = test[['PassengerId', 'Survived']]

submission.to_csv("submission.csv", index=False)

print("Submission file created: submission.csv")

Submission file created: submission.csv
